# JHU Hub: collect AI archive links

Walks `https://hub.jhu.edu/tags/artificial-intelligence/articles/` page by page and extracts each archive card's title, Hub URL, displayed date, category label, and summary. Stops when a page yields no new URLs.

Output: `./Data/JHU Hub Articles Links.csv`

Some entries are *Hopkins in the news* links that redirect to external publishers; they are kept in the links file and handled in the scraping/cleanup notebooks.


In [ ]:
from pathlib import Path
from time import sleep
import random
import re
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://hub.jhu.edu"
START_URL = f"{BASE_URL}/tags/artificial-intelligence/articles/"

DATA_DIR = Path("./Data")
OUTPUT_PATH = DATA_DIR / "JHU Hub Articles Links.csv"

MAX_PAGES = 500
REQUEST_TIMEOUT = 30
DELAY_RANGE = (0.8, 1.5)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# Visible dates currently include both ordinary dates such as
# "July 2, 2026" and magazine issue dates such as "Summer 2026".
MONTH_OR_SEASON = (
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|"
    r"Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?|"
    r"Spring|Summer|Fall|Autumn|Winter)"
)
PUBLISHED_DATE_RE = re.compile(
    rf"\bPublished\s+({MONTH_OR_SEASON}\s+(?:\d{{1,2}},\s+)?\d{{4}})\b",
    flags=re.IGNORECASE,
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Persistent session with retry handling.
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
def clean_text(value):
    """Collapse repeated whitespace and return a clean string."""
    if value is None:
        return ""
    return " ".join(str(value).split())


def canonicalize_hub_url(href):
    """Resolve and normalize a Hub URL while removing query strings/fragments."""
    absolute = urljoin(BASE_URL, href)
    parts = urlsplit(absolute)

    if parts.netloc.lower() != "hub.jhu.edu":
        return ""

    path = re.sub(r"/{2,}", "/", parts.path)
    if not path.endswith("/"):
        path += "/"

    return urlunsplit(("https", "hub.jhu.edu", path, "", ""))


def page_url(page_number):
    """The first archive page has no page number; page 2 starts at /articles/2/."""
    if page_number == 1:
        return START_URL
    return urljoin(START_URL, f"{page_number}/")


def is_navigation_url(url):
    """Reject archive/navigation links that can appear in headings."""
    path = urlsplit(url).path.lower()

    blocked_prefixes = (
        "/tags/",
        "/topics/",
        "/events/",
        "/about/",
        "/search/",
        "/announcements/",
    )

    return (
        path == "/"
        or path.startswith(blocked_prefixes)
        or path.rstrip("/") in {
            "/at-work",
            "/magazine",
            "/contact",
            "/for-media",
        }
    )


def nearest_archive_card(heading):
    """Find the nearest ancestor whose text contains the card's Published label."""
    for parent in heading.parents:
        if getattr(parent, "name", None) in {"main", "body", "html"}:
            break

        text = clean_text(parent.get_text(" ", strip=True))
        if PUBLISHED_DATE_RE.search(text):
            return parent

    return heading.parent


def extract_category(card, heading, title, published_date, summary):
    """Extract the short label displayed above the archive-card headline."""
    category_selectors = [
        ".topic",
        ".category",
        ".tag",
        ".eyebrow",
        ".kicker",
        "[class*='topic']",
        "[class*='category']",
        "[class*='eyebrow']",
        "[class*='kicker']",
    ]

    for selector in category_selectors:
        for node in card.select(selector):
            text = clean_text(node.get_text(" ", strip=True))
            if text and text not in {title, published_date, summary}:
                return text

    # Fallback: inspect short elements appearing before the heading.
    for node in card.find_all(["span", "p", "h6"]):
        if node is heading or heading in getattr(node, "descendants", []):
            continue

        text = clean_text(node.get_text(" ", strip=True))
        if not text:
            continue
        if text in {title, summary} or "Published" in text:
            continue
        if len(text) <= 80:
            return text

    return ""


def extract_summary(card, title, published_date):
    """Choose the most plausible descriptive paragraph from one archive card."""
    candidates = []

    for node in card.find_all("p"):
        text = clean_text(node.get_text(" ", strip=True))
        if not text:
            continue
        if text == title or text == published_date:
            continue
        if PUBLISHED_DATE_RE.search(text):
            continue
        if text.lower().startswith(("published ", "posted ")):
            continue
        candidates.append(text)

    if not candidates:
        return ""

    # Descriptions are normally the longest paragraph in a card.
    return max(candidates, key=len)


def extract_archive_records(soup, archive_page_url):
    """Extract article-card records without depending on one fragile CSS class."""
    main = soup.select_one("main") or soup
    records_by_url = {}

    for heading in main.find_all(["h2", "h3", "h4", "h5", "h6"]):
        anchor = heading.find("a", href=True)
        if anchor is None:
            continue

        url = canonicalize_hub_url(anchor.get("href", ""))
        if not url or is_navigation_url(url):
            continue

        card = nearest_archive_card(heading)
        card_text = clean_text(card.get_text(" ", strip=True))
        date_match = PUBLISHED_DATE_RE.search(card_text)

        # Article cards on this archive contain an explicit Published label.
        if not date_match:
            continue

        title = clean_text(anchor.get_text(" ", strip=True))
        if not title:
            image = anchor.find("img")
            title = clean_text(image.get("alt", "")) if image else ""

        if not title:
            continue

        published_date = clean_text(date_match.group(1))
        summary = extract_summary(card, title, published_date)
        category = extract_category(card, heading, title, published_date, summary)

        record = {
            "Article": title,
            "Link": url,
            "Listing Category": category,
            "Listing Published Date": published_date,
            "Listing Summary": summary,
            "Archive Page": archive_page_url,
        }

        old = records_by_url.get(url)
        if old is None or len(record["Listing Summary"]) > len(old["Listing Summary"]):
            records_by_url[url] = record

    return list(records_by_url.values())


def fetch_archive_page(page_number):
    url = page_url(page_number)
    response = session.get(url, timeout=REQUEST_TIMEOUT)

    if response.status_code == 404:
        return None, url, response.status_code

    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser"), response.url, response.status_code

In [ ]:
# -----------------------------
# Crawl all archive pages
# -----------------------------
all_records = []
seen_urls = set()

for page_number in tqdm(range(1, MAX_PAGES + 1), desc="Archive pages"):
    try:
        soup, fetched_url, status_code = fetch_archive_page(page_number)
    except requests.RequestException as exc:
        print(f"Stopped at archive page {page_number}: {exc}")
        break

    if soup is None:
        print(f"Stopped: archive page {page_number} returned HTTP {status_code}.")
        break

    page_records = extract_archive_records(soup, fetched_url)
    new_records = [row for row in page_records if row["Link"] not in seen_urls]

    # Out-of-range pages may be empty or may repeat the final valid page.
    if not new_records:
        print(f"Stopped at archive page {page_number}: no new article links found.")
        break

    for row in new_records:
        row["Page Number"] = page_number
        all_records.append(row)
        seen_urls.add(row["Link"])

    checkpoint = pd.DataFrame(all_records)
    checkpoint.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print(
        f"Archive page {page_number}: {len(new_records)} new links; "
        f"{len(all_records)} unique links in total."
    )

    sleep(random.uniform(*DELAY_RANGE))

In [ ]:
article_links = (
    pd.DataFrame(all_records)
    .drop_duplicates(subset="Link")
    .reset_index(drop=True)
)

article_links.head()

In [ ]:
article_links.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved {len(article_links)} unique archive entries to: {OUTPUT_PATH.resolve()}")